# Demo: Context-Free Grammar và Constituency Parsing

Notebook này minh họa cách dùng **Context-Free Grammar (CFG)** để phân tích cấu trúc cú pháp của câu bằng **Constituency Parsing**.

Thay vì chỉ tách câu thành từng từ, constituency parsing tìm cách nhóm các từ thành những cụm có chức năng ngữ pháp như:

- `NP`: noun phrase / cụm danh từ
- `VP`: verb phrase / cụm động từ
- `PP`: prepositional phrase / cụm giới từ
- `S`: sentence / câu hoàn chỉnh

Trong demo này, ta dùng các câu tiếng Việt đơn giản thuộc ngữ cảnh **âm nhạc** để dễ quan sát cấu trúc câu.


## 1. Cài đặt và import thư viện

Ta sử dụng thư viện `nltk`. Phần này chỉ cần các lớp `CFG` và parser có sẵn, không cần tải thêm corpus.


In [1]:
# Nếu chạy trên Google Colab mà chưa có nltk, bỏ dấu # dòng dưới:
!pip install nltk

import nltk
from nltk import CFG
from nltk.parse import ChartParser


## 2. Ý tưởng của Context-Free Grammar

CFG là một tập các luật sinh dùng để mô tả cấu trúc câu.

Ví dụ:

```text
S  -> NP VP
NP -> Det N
VP -> V NP
```

Có thể hiểu đơn giản:

- Một câu `S` gồm một cụm danh từ `NP` và một cụm động từ `VP`.
- Một cụm danh từ `NP` có thể gồm từ chỉ định `Det` và danh từ `N`.
- Một cụm động từ `VP` có thể gồm động từ `V` và một cụm danh từ `NP`.

Parser sẽ dùng các luật này để kiểm tra câu có hợp lệ theo ngữ pháp hay không, đồng thời dựng cây cú pháp cho câu.


## 3. Xây dựng bộ ngữ pháp CFG

Bộ ngữ pháp dưới đây được thiết kế cho một số câu tiếng Việt đơn giản về âm nhạc.

Ví dụ câu có thể phân tích:

```text
người_nghe thích bài_hát
ca_sĩ hát bản_nhạc trong phòng_thu
ban_nhạc chơi nhạc với cây_đàn
```


In [2]:
grammar = CFG.fromstring("""
S -> NP VP

NP -> N
NP -> Det N
NP -> Adj N
NP -> Det Adj N
NP -> NP PP

VP -> V
VP -> V NP
VP -> V PP
VP -> V NP PP
VP -> Adv VP

PP -> P NP

Det -> 'một' | 'những' | 'bài' | 'bản' | 'cây'
N -> 'người_nghe' | 'ca_sĩ' | 'ban_nhạc' | 'bài_hát' | 'bản_nhạc' | 'giai_điệu' | 'cây_đàn' | 'phòng_thu' | 'sân_khấu' | 'album'
Adj -> 'hay' | 'mới' | 'buồn' | 'nhẹ_nhàng'
V -> 'thích' | 'nghe' | 'hát' | 'chơi' | 'thu_âm' | 'phát_hành'
P -> 'trong' | 'trên' | 'với'
Adv -> 'đang' | 'rất'
""")

print(grammar)


Grammar with 42 productions (start state = S)
    S -> NP VP
    NP -> N
    NP -> Det N
    NP -> Adj N
    NP -> Det Adj N
    NP -> NP PP
    VP -> V
    VP -> V NP
    VP -> V PP
    VP -> V NP PP
    VP -> Adv VP
    PP -> P NP
    Det -> 'một'
    Det -> 'những'
    Det -> 'bài'
    Det -> 'bản'
    Det -> 'cây'
    N -> 'người_nghe'
    N -> 'ca_sĩ'
    N -> 'ban_nhạc'
    N -> 'bài_hát'
    N -> 'bản_nhạc'
    N -> 'giai_điệu'
    N -> 'cây_đàn'
    N -> 'phòng_thu'
    N -> 'sân_khấu'
    N -> 'album'
    Adj -> 'hay'
    Adj -> 'mới'
    Adj -> 'buồn'
    Adj -> 'nhẹ_nhàng'
    V -> 'thích'
    V -> 'nghe'
    V -> 'hát'
    V -> 'chơi'
    V -> 'thu_âm'
    V -> 'phát_hành'
    P -> 'trong'
    P -> 'trên'
    P -> 'với'
    Adv -> 'đang'
    Adv -> 'rất'


## 4. Tạo parser

Ở đây ta dùng `ChartParser` vì nó phù hợp để phân tích câu theo CFG và có thể trả về nhiều cây nếu câu có nhiều cách hiểu.


In [3]:
parser = ChartParser(grammar)
print("Đã tạo ChartParser thành công.")


Đã tạo ChartParser thành công.


## 5. Hàm phân tích một câu

Hàm dưới đây nhận một câu đã được tách token bằng dấu cách, sau đó in ra các cây cú pháp tìm được.

Lưu ý: trong demo này, các từ ghép như `bài hát`, `ban nhạc`, `phòng thu` được viết thành `bài_hát`, `ban_nhạc`, `phòng_thu` để parser xem chúng như một token.


In [5]:
def parse_sentence(sentence):
    tokens = sentence.split()
    print("Câu đầu vào:", sentence)
    print("Tokens:", tokens)
    print("-" * 60)

    trees = list(parser.parse(tokens))

    if len(trees) == 0:
        print("Không tìm thấy cây cú pháp phù hợp với grammar hiện tại.")
        return []

    print(f"Số cây cú pháp tìm được: {len(trees)}")
    for i, tree in enumerate(trees, 1):
        print(f"Cây cú pháp {i}:")
        print(tree)
        print("Dạng phân cấp:")
        tree.pretty_print()

    return trees


## 6. Phân tích các câu mẫu

### Ví dụ 1: Câu đơn giản

Câu này có cấu trúc cơ bản:

```text
S -> NP VP
```

Trong đó:

- `người_nghe` là `NP`
- `thích bài_hát` là `VP`


In [ ]:
trees_1 = parse_sentence("người_nghe thích bài_hát")


### Ví dụ 2: Câu có trạng từ

Câu dưới đây có thêm trạng từ `đang`, làm cho cụm động từ phức tạp hơn.


In [6]:
trees_2 = parse_sentence("ca_sĩ đang hát bản_nhạc")


Câu đầu vào: ca_sĩ đang hát bản_nhạc
Tokens: ['ca_sĩ', 'đang', 'hát', 'bản_nhạc']
------------------------------------------------------------
Số cây cú pháp tìm được: 1
Cây cú pháp 1:
(S (NP (N ca_sĩ)) (VP (Adv đang) (VP (V hát) (NP (N bản_nhạc)))))
Dạng phân cấp:
            S                  
   _________|___                
  |             VP             
  |     ________|___            
  |    |            VP         
  |    |         ___|_____      
  NP   |        |         NP   
  |    |        |         |     
  N   Adv       V         N    
  |    |        |         |     
ca_sĩ đang     hát     bản_nhạc



### Ví dụ 3: Câu có cụm giới từ

Cụm `trong phòng_thu` là một `PP`, bổ sung thông tin về nơi diễn ra hành động.


In [7]:
trees_3 = parse_sentence("ca_sĩ hát bản_nhạc trong phòng_thu")


Câu đầu vào: ca_sĩ hát bản_nhạc trong phòng_thu
Tokens: ['ca_sĩ', 'hát', 'bản_nhạc', 'trong', 'phòng_thu']
------------------------------------------------------------
Số cây cú pháp tìm được: 2
Cây cú pháp 1:
(S
  (NP (N ca_sĩ))
  (VP (V hát) (NP (N bản_nhạc)) (PP (P trong) (NP (N phòng_thu)))))
Dạng phân cấp:
       S                              
   ____|_____                          
  |          VP                       
  |     _____|____________             
  |    |     |            PP          
  |    |     |        ____|______      
  NP   |     NP      |           NP   
  |    |     |       |           |     
  N    V     N       P           N    
  |    |     |       |           |     
ca_sĩ hát bản_nhạc trong     phòng_thu

Cây cú pháp 2:
(S
  (NP (N ca_sĩ))
  (VP
    (V hát)
    (NP (NP (N bản_nhạc)) (PP (P trong) (NP (N phòng_thu))))))
Dạng phân cấp:
             S                            
   __________|______                       
  |                 VP            

## 7. Ambiguity trong constituency parsing

Một điểm thú vị của CFG là một câu có thể có nhiều cây cú pháp khác nhau.

Ví dụ câu:

```text
ban_nhạc chơi nhạc với cây_đàn
```

Cụm `với cây_đàn` có thể được hiểu theo hai cách:

1. Gắn với động từ `chơi`: ban nhạc chơi bằng cây đàn.
2. Gắn với cụm danh từ phía trước: nhạc với cây đàn.

Đây là hiện tượng **syntactic ambiguity**.


In [9]:
ambiguous_grammar = CFG.fromstring("""
S -> NP VP

NP -> N
NP -> Det N
NP -> NP PP

VP -> V NP
VP -> V NP PP

PP -> P NP

Det -> 'cây'
N -> 'ban_nhạc' | 'nhạc' | 'cây_đàn'
V -> 'chơi'
P -> 'với'
""")

ambiguous_parser = ChartParser(ambiguous_grammar)

def parse_ambiguous(sentence):
    tokens = sentence.split()
    trees = list(ambiguous_parser.parse(tokens))
    print("Câu đầu vào:", sentence)
    print("Số cây cú pháp tìm được:", len(trees))
    print("-" * 60)

    for i, tree in enumerate(trees, 1):
        print(f"Cây cú pháp {i}:")
        print(tree)
        tree.pretty_print()

    return trees

ambiguous_trees = parse_ambiguous("ban_nhạc chơi nhạc với cây_đàn")


Câu đầu vào: ban_nhạc chơi nhạc với cây_đàn
Số cây cú pháp tìm được: 2
------------------------------------------------------------
Cây cú pháp 1:
(S
  (NP (N ban_nhạc))
  (VP (V chơi) (NP (N nhạc)) (PP (P với) (NP (N cây_đàn)))))
          S                       
    ______|____                    
   |           VP                 
   |       ____|________           
   |      |    |        PP        
   |      |    |     ___|_____     
   NP     |    NP   |         NP  
   |      |    |    |         |    
   N      V    N    P         N   
   |      |    |    |         |    
ban_nhạc chơi nhạc với     cây_đàn

Cây cú pháp 2:
(S
  (NP (N ban_nhạc))
  (VP (V chơi) (NP (NP (N nhạc)) (PP (P với) (NP (N cây_đàn))))))
               S                      
    ___________|____                   
   |                VP                
   |       _________|___               
   |      |             NP            
   |      |     ________|___           
   |      |    |            PP       

## 8. Kiểm tra câu không hợp lệ với grammar

Nếu câu chứa từ chưa có trong grammar hoặc cấu trúc không khớp luật sinh, parser sẽ không tạo được cây cú pháp.


In [10]:
trees_invalid = parse_sentence("album nghe ca_sĩ")


Câu đầu vào: album nghe ca_sĩ
Tokens: ['album', 'nghe', 'ca_sĩ']
------------------------------------------------------------
Số cây cú pháp tìm được: 1
Cây cú pháp 1:
(S (NP (N album)) (VP (V nghe) (NP (N ca_sĩ))))
Dạng phân cấp:
       S            
   ____|____         
  |         VP      
  |     ____|____    
  NP   |         NP 
  |    |         |   
  N    V         N  
  |    |         |   
album nghe     ca_sĩ



## 9. Trích xuất thông tin từ cây cú pháp

Sau khi có cây cú pháp, ta có thể duyệt cây để lấy ra các thành phần như `NP`, `VP`, `PP`.


In [11]:
def extract_phrases(tree, label):
    phrases = []
    for subtree in tree.subtrees(lambda t: t.label() == label):
        phrase = " ".join(subtree.leaves())
        phrases.append(phrase)
    return phrases

if trees_3:
    tree = trees_3[0]
    print("Câu:", " ".join(tree.leaves()))
    print("Các NP:", extract_phrases(tree, "NP"))
    print("Các VP:", extract_phrases(tree, "VP"))
    print("Các PP:", extract_phrases(tree, "PP"))


Câu: ca_sĩ hát bản_nhạc trong phòng_thu
Các NP: ['ca_sĩ', 'bản_nhạc', 'phòng_thu']
Các VP: ['hát bản_nhạc trong phòng_thu']
Các PP: ['trong phòng_thu']


## 10. Tự nhập câu để kiểm tra

Bạn có thể sửa biến `custom_sentence` bằng các từ đã có trong grammar.


In [12]:
custom_sentence = "ban_nhạc chơi cây_đàn trên sân_khấu"
custom_trees = parse_sentence(custom_sentence)


Câu đầu vào: ban_nhạc chơi cây_đàn trên sân_khấu
Tokens: ['ban_nhạc', 'chơi', 'cây_đàn', 'trên', 'sân_khấu']
------------------------------------------------------------
Số cây cú pháp tìm được: 2
Cây cú pháp 1:
(S
  (NP (N ban_nhạc))
  (VP (V chơi) (NP (N cây_đàn)) (PP (P trên) (NP (N sân_khấu)))))
Dạng phân cấp:
          S                            
    ______|______                       
   |             VP                    
   |       ______|__________            
   |      |      |          PP         
   |      |      |      ____|_____      
   NP     |      NP    |          NP   
   |      |      |     |          |     
   N      V      N     P          N    
   |      |      |     |          |     
ban_nhạc chơi cây_đàn trên     sân_khấu

Cây cú pháp 2:
(S
  (NP (N ban_nhạc))
  (VP (V chơi) (NP (NP (N cây_đàn)) (PP (P trên) (NP (N sân_khấu))))))
Dạng phân cấp:
                 S                         
    _____________|_____                     
   |                   VP

## 11. Tóm tắt pipeline

Pipeline của demo này gồm các bước:

```text
Viết grammar CFG
→ Tạo parser
→ Tách câu thành token
→ Parser tìm cây cú pháp
→ Hiển thị constituency tree
→ Phân tích các cụm NP, VP, PP
```

Kết luận:

- CFG giúp mô tả cấu trúc câu bằng các luật sinh rõ ràng.
- Constituency parsing cho biết câu được tạo thành từ những cụm cú pháp nào.
- Một câu có thể có nhiều cây cú pháp nếu có nhiều cách hiểu.
- Hạn chế của cách làm này là phải tự viết grammar, nên khó bao phủ toàn bộ ngôn ngữ tự nhiên.
